# Lab 12 solution, notebook 2: choose the filter, and check the cited files

The worked answers to the two challenge tasks of `12b_retrieval_at_the_desk.ipynb`. The setup repeats what the tasks
need from the notebook (the 36 files in a collection, `embed`, `retrieve` and `answer`), so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import chromadb
from openai import OpenAI, omit

client = OpenAI(timeout=60)
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]

files = sorted((ROOT / "data" / "corpus").glob("*.md"))
names = [p.name for p in files]
texts = [p.read_text() for p in files]
metadatas = [{"kind": name.split("_")[0]} for name in names]
vectors = [item.embedding for item in client.embeddings.create(model=EMBEDDING_MODEL, input=texts).data]

collection = chromadb.EphemeralClient().get_or_create_collection(
    "shop_files", configuration={"hnsw": {"space": "cosine"}})
collection.add(ids=names, documents=texts, metadatas=metadatas, embeddings=vectors)
print("stored:", collection.count())

stored: 36


The same functions as the notebook. `answer` takes the rules as an argument here, so Task 2 can pass new ones.

In [3]:
RULES = ("Answer the question using only the documents below. "
         "If they do not contain the answer, say you do not know.")


def embed(text):
    return client.embeddings.create(model=EMBEDDING_MODEL, input=text).data[0].embedding


def retrieve(question, k=3, where=None):
    got = collection.query(query_embeddings=[embed(question)], n_results=k, where=where)
    rows = []
    for file_id, text, distance in zip(got["ids"][0], got["documents"][0], got["distances"][0]):
        rows.append({"id": file_id, "text": text, "cosine": round(1 - distance, 4)})
    return rows


def answer(question, k=3, where=None, rules=RULES):
    hits = retrieve(question, k, where)
    packed = "\n\n".join(f"[{row['id']}]\n{row['text']}" for row in hits)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": rules},
            {"role": "user", "content": f"Documents:\n\n{packed}\n\nQuestion: {question}"},
        ],
        max_completion_tokens=200,
        reasoning_effort=EFFORT or omit,
    )
    return [row["id"] for row in hits], response.choices[0].message.content, response.usage.prompt_tokens

---

## Task 1: Choose the filter that can answer

In [4]:
SHIPPED = "When was the replacement for the warped vinyl shipped?"

for where in [{"kind": "ticket"}, {"kind": "policy"}]:
    used, reply, _ = answer(SHIPPED, k=1, where=where)
    print("where =", where, "| files used:", used)
    print("  ", reply)

where = {'kind': 'ticket'} | files used: ['ticket_06_germany_warped_vinyl.md']
   The replacement was shipped on 2012-07-20.


where = {'kind': 'policy'} | files used: ['policy_damaged_media.md']
   I don’t know. The document says warped vinyl is eligible for a free replacement, but it doesn’t say when it was shipped.


With the tickets-only filter the search used `ticket_06_germany_warped_vinyl.md`, and the reply gave the date: *The replacement was shipped on 2012-07-20.* With the policies-only filter it used `policy_damaged_media.md`, and the reply said it did not know when the replacement was shipped.

The tickets-only filter was the right one: the date is a fact about one case, and cases are tickets. A filter is a
choice about where the answer lives, made before the search runs.

---

## Task 2: Make the reply cite its files, and check the citations in code

The new rules add one sentence: end with a `Sources:` line.

In [5]:
CITED_RULES = RULES + (" End your reply with a line 'Sources:' followed by the file names you used, "
                       "separated by commas, or 'Sources: none' if you used none.")

The check takes the text after the last `Sources:` in the reply (`rsplit("Sources:", 1)` splits once, from the
right), splits it at the commas, and keeps every name that is not `none`. It does not need `Sources:` to start a line:
a model can put it at the end of a sentence. A citation passes when the name is in the list of files the search
returned. `strip(" []")` removes spaces and square brackets, in case the model copies the brackets of the packed
prompt.

In [6]:
def cited_files(reply):
    if "Sources:" not in reply:
        return None
    names_cited = reply.rsplit("Sources:", 1)[1].split(",")
    return [name.strip(" []") for name in names_cited if name.strip(" []").lower() not in ("", "none")]


for question, k in [("How long do I have to return an unopened CD?", 3), ("When is support staffed?", 1)]:
    used, reply, _ = answer(question, k=k, rules=CITED_RULES)
    cited = cited_files(reply)
    print(question, "| k =", k)
    print("  reply:", reply.replace("\n", " / "))
    print("  retrieved:", used)
    if cited is None:
        print("  ❌ no Sources line")
    else:
        unknown = [name for name in cited if name not in used]
        verdict = "✅ every cited file was retrieved" if not unknown else f"❌ not retrieved: {unknown}"
        print("  cited:", cited, "|", verdict)

How long do I have to return an unopened CD? | k = 3
  reply: You have 30 days from the invoice date to return an unopened CD. /  / Sources: policy_returns.md
  retrieved: ['policy_returns.md', 'policy_warranty.md', 'policy_damaged_media.md']
  cited: ['policy_returns.md'] | ✅ every cited file was retrieved


When is support staffed? | k = 1
  reply: I don’t know. The document names support representatives but does not say when support is staffed. /  / Sources: policy_support_reps.md
  retrieved: ['policy_support_reps.md']
  cited: ['policy_support_reps.md'] | ✅ every cited file was retrieved


Both replies end with a `Sources:` line, and the check passes on both: the returns reply cites `policy_returns.md`, and the staffing reply at `k=1` cites `policy_support_reps.md`.

**What the check establishes:** every cited name is a file the model was given. It does not establish that the cited
file says what the reply claims; for that, a person (or a second check) reads the file. And in the staffing case the check passes while the reply is no help to the rep: the cited file is real and was retrieved, and the search had still missed the file with the hours.